In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

C:\Users\Rudroju Manideep\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [6]:
import os
os.chdir("../")

In [7]:
extracted_data = load_pdf_files("data")

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
def text_split(minimal_docs: List[Document]) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [12]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings
embeddings = download_embeddings()

C:\Users\Rudroju Manideep\AppData\Local\Temp\ipykernel_8560\4014230296.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [14]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
vector = embeddings.embed_query("What is the capital of France?")
vector

[0.08204811066389084,
 0.03605553135275841,
 -0.0038928852882236242,
 -0.0048810457810759544,
 0.02565113641321659,
 -0.05714348703622818,
 0.012191606685519218,
 0.004678904078900814,
 0.03494987264275551,
 -0.0224219411611557,
 -0.008005237206816673,
 -0.10935354232788086,
 0.022724784910678864,
 -0.02932087890803814,
 -0.04352205619215965,
 -0.12024123221635818,
 -0.000848641328047961,
 -0.018150122836232185,
 0.056129537522792816,
 0.003085229778662324,
 0.0023363472428172827,
 -0.01683923974633217,
 0.06362469494342804,
 -0.023660214617848396,
 0.03149356320500374,
 -0.034797921776771545,
 -0.0205488633364439,
 -0.002790951170027256,
 -0.011037975549697876,
 -0.03612672537565231,
 0.0541410930454731,
 -0.036617133766412735,
 -0.02500864863395691,
 -0.03817041590809822,
 -0.04960364103317261,
 -0.015148096717894077,
 0.02131503075361252,
 -0.012740420177578926,
 0.07670091837644577,
 0.04435573145747185,
 -0.010834861546754837,
 -0.029760034754872322,
 -0.016970466822385788,
 -0.02

In [16]:
print(len(vector))

384


In [99]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [100]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [101]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [102]:
pc

In [103]:
from pinecone import ServerlessSpec

index_name = "cureai-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [104]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [105]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name
)

In [106]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [107]:
retrieved_docs = retriever.invoke("what is acne?")
retrieved_docs

[Document(id='98edcba6-a02d-48b0-9800-699c95b4e8a9', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='577f07b0-c1a3-4fd1-971d-ee5329391a3c', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='2ac43324-0bc3-4cf6-84cd-ad9b00d8ae49', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [108]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [109]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [110]:
system_prompt = (
    "You are a helpful assistant for answering questions about medical conditions. Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know. Use three sentences maximum to answer the question."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [111]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [112]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris, the medical term for common acne, is the most common skin disease.


In [113]:
response = rag_chain.invoke({"input": "What is abuse?"})
print(response["answer"])

Abuse is defined as any harmful, injurious, or offensive action, including excessive and wrongful misuse of anything. There are several major types of abuse, including physical and sexual abuse of a child or an adult, substance abuse, elderly abuse, and emotional abuse. Abuse can take many forms and can be inflicted by anyone.
